<a href="https://colab.research.google.com/github/carlosss1111/Financial_Asset_Risk_Management-Trading_Signals_Model/blob/main/Financial_Asset_Risk_Management_BTCvsETH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ========================
# - Hace: baseline IS, subperiodos, walk-forward OOS, robustez (grid),
#         bootstrap por bloques (CI Sharpe/CAGR/mean), White Reality Check (multiple testing),
#         riesgo (Sortino/Calmar/VaR/CVaR), alpha vs benchmark (OLS+NW),
#         ann_factor automático (≈365 cripto; ≈252 equity y mixtos por intersección),
#         y guarda CSV/PNG en outdir.
#


# Timing (anti look-ahead): decisión con stress hasta cierre de t, aplica retorno t->t+1.
# =========================



!pip -q install numpy pandas matplotlib pandas_datareader scipy

import os, sys, math, argparse, warnings
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pandas_datareader import data as pdr
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# -----------------------------
# Notebook-safe
# -----------------------------
def _in_notebook() -> bool:
    return ("ipykernel" in sys.modules) or ("COLAB_GPU" in os.environ) or ("JPY_PARENT_PID" in os.environ)

def _parse_known_notebook_safe(ap: argparse.ArgumentParser, argv: Optional[List[str]]):
    if argv is None:
        argv = [] if _in_notebook() else sys.argv[1:]
    args, _unknown = ap.parse_known_args(argv)  # ignora -f kernel.json, etc.
    return args



# -----------------------------
# Robust FRED download (DataReader -> CSV fallback)
# -----------------------------
def load_fred_series(series_id: str, start: str, end: str) -> pd.Series:
    # 1) DataReader
    try:
        df = pdr.DataReader(series_id, "fred", start, end)
        s = df[series_id].copy()
        s.index = pd.to_datetime(s.index)
        return s.dropna()
    except Exception:
        pass

    # 2) Public CSV fallback
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)
    if "DATE" not in df.columns or series_id not in df.columns:
        raise RuntimeError(f"CSV fallback failed for {series_id} (unexpected format).")
    df["DATE"] = pd.to_datetime(df["DATE"])
    df = df.set_index("DATE").sort_index()
    s = pd.to_numeric(df[series_id], errors="coerce").dropna()
    if start:
        s = s[s.index >= pd.to_datetime(start)]
    if end:
        s = s[s.index <= pd.to_datetime(end)]
    return s.dropna()

def load_pair_prices(series_a: str, series_b: str, start: str, end: str, cache_dir: str = "fred_cache") -> pd.DataFrame:
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"{series_a}__{series_b}__{start}__{end}.csv")
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        df["DATE"] = pd.to_datetime(df["DATE"])
        df = df.set_index("DATE").sort_index()
        return df[[series_a, series_b]].dropna()

    a = load_fred_series(series_a, start, end).rename(series_a)
    b = load_fred_series(series_b, start, end).rename(series_b)
    df = pd.concat([a, b], axis=1).dropna()

    out = df.copy()
    out.index.name = "DATE"
    out.reset_index().to_csv(cache_path, index=False)
    return df

def pick_gold_series(start: str, end: str) -> Optional[str]:
    candidates = ["GOLDAMGBD228NLBM", "GOLDPMGBD228NLBM"]
    for s in candidates:
        try:
            _ = load_fred_series(s, start, end)
            return s
        except Exception:
            continue
    return None

# ----------------------------------
# Utils
# ---------------------------------------
def clip01(x: np.ndarray) -> np.ndarray:
    return np.minimum(1.0, np.maximum(0.0, x))

def safe_name(s: str) -> str:
    bad = [" ", "/", "\\", "(", ")", "{", "}", "[", "]", "≡", "–", "—", ":", ";", ",", "|", "$", "^"]
    out = s
    for b in bad:
        out = out.replace(b, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")

def estimate_ann_factor(dates: pd.DatetimeIndex) -> float:
    if len(dates) < 50:
        return 252.0
    years = pd.Series(dates.year, index=dates)
    counts = years.groupby(years).size().values
    if len(counts) == 0:
        return 252.0
    af = float(np.median(counts))
    af = max(200.0, min(366.0, af))
    return af

def nw_hac_tstat(x: np.ndarray, max_lag: Optional[int] = None) -> Tuple[float, float]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 5:
        return (np.nan, np.nan)
    mu = x.mean()
    u = x - mu
    if max_lag is None:
        max_lag = int(np.floor(4.0 * (n / 100.0) ** (2.0 / 9.0)))
        max_lag = max(0, max_lag)

    gamma0 = np.dot(u, u) / n
    s = gamma0
    for k in range(1, max_lag + 1):
        w = 1.0 - k / (max_lag + 1.0)
        gk = np.dot(u[k:], u[:-k]) / n
        s += 2.0 * w * gk

    var_mean = s / n
    if var_mean <= 0:
        return (np.nan, np.nan)
    se = math.sqrt(var_mean)
    t = mu / se
    return (t, se)



def nw_ols_alpha_beta(y: np.ndarray, x: np.ndarray, add_const: bool = True, max_lag: Optional[int] = None):


    y = np.asarray(y, float)
    x = np.asarray(x, float)
    mask = np.isfinite(y) & np.isfinite(x)
    y = y[mask]; x = x[mask]
    n = len(y)
    if n < 20:
        return dict(alpha=np.nan, beta=np.nan, t_alpha=np.nan, se_alpha=np.nan)

    if add_const:
        X = np.column_stack([np.ones(n), x])
    else:
        X = x.reshape(-1, 1)

    # OLS
    XtX = X.T @ X
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)
    b = XtX_inv @ (X.T @ y)
    yhat = X @ b
    u = y - yhat

    # NW for covariance of b
    if max_lag is None:
        max_lag = int(np.floor(4.0 * (n / 100.0) ** (2.0 / 9.0)))
        max_lag = max(0, max_lag)

    S = np.zeros((X.shape[1], X.shape[1]), float)
    # lag 0
    for t in range(n):
        xt = X[t:t+1].T
        S += (u[t]**2) * (xt @ xt.T)
    # lags
    for k in range(1, max_lag + 1):
        w = 1.0 - k / (max_lag + 1.0)
        for t in range(k, n):
            xt = X[t:t+1].T
            xlag = X[t-k:t-k+1].T
            S += w * u[t] * u[t-k] * (xt @ xlag.T + xlag @ xt.T)

    cov_b = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.maximum(0.0, np.diag(cov_b)))

    if add_const:
        alpha = float(b[0]); beta = float(b[1])
        se_alpha = float(se[0]) if len(se) > 0 else np.nan
        t_alpha = float(alpha / se_alpha) if se_alpha and se_alpha > 0 else np.nan
    else:
        alpha = 0.0; beta = float(b[0])
        se_alpha = np.nan; t_alpha = np.nan

    return dict(alpha=alpha, beta=beta, t_alpha=t_alpha, se_alpha=se_alpha)

def max_drawdown(equity: pd.Series) -> float:
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return float(dd.min())

def downside_dev(r: np.ndarray) -> float:
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    if len(r) < 2:
        return np.nan
    neg = np.minimum(r, 0.0)
    return float(np.sqrt(np.mean(neg**2)))

def var_cvar(r: np.ndarray, alpha: float = 0.05) -> Tuple[float, float]:
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    if len(r) < 10:
        return (np.nan, np.nan)
    q = float(np.quantile(r, alpha))
    tail = r[r <= q]
    cvar = float(tail.mean()) if len(tail) > 0 else np.nan
    return (q, cvar)

def annualize_turnover(turnover: float, n_days: int, ann_factor: float) -> float:
    if n_days <= 0:
        return np.nan
    return float(turnover) * (ann_factor / float(n_days))



# -----------------------------
# THFE from weighted quantiles
# -----------------------------
def weighted_quantile_step(values: np.ndarray, weights: np.ndarray, probs: List[float]) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    v = values[order]
    w = weights[order]
    cw = np.cumsum(w) / w.sum()
    out = []
    for p in probs:
        i = int(np.searchsorted(cw, p, side="left"))
        i = min(i, len(v) - 1)
        out.append(v[i])
    return np.array(out, dtype=float)

def make_thfe(stress_window_most_recent_first: np.ndarray,
             lam: float,
             probs: Tuple[float, ...],
             dedup: bool = True,
             round_decimals: int = 12) -> List[float]:
    H = len(stress_window_most_recent_first)
    w = np.array([lam ** k for k in range(H)], dtype=float)
    q = weighted_quantile_step(stress_window_most_recent_first, w, list(probs))
    if round_decimals is not None:
        q = np.round(q, round_decimals)
    return sorted(set(q.tolist())) if dedup else sorted(q.tolist())



# ------------------------------------------
# Comparators
# ---------------------------------------

WeightFamily = Callable[[int], List[int]]

def w_uniform(n: int) -> List[int]:
    return [1] * n

def w_centered_triangular(n: int) -> List[int]:
    return [1 + min(i, n - 1 - i) for i in range(n)]

def w_tail_base_n(n: int) -> List[int]:
    return [pow(n, i) for i in range(n)]

def compare_padding(A: List[float], B: List[float], mode: str, tol: float = 0.0) -> Optional[int]:
    a = sorted(A); b = sorted(B)
    k = max(len(a), len(b))

    def pad(x: List[float], target: int) -> List[float]:
        if len(x) == target:
            return list(x)
        if mode == "optimistic":
            fill = max(x)
            return list(x) + [fill] * (target - len(x))
        elif mode == "pessimistic":
            fill = min(x)
            return [fill] * (target - len(x)) + list(x)
        else:
            raise ValueError("mode must be optimistic/pessimistic")

    ap = pad(a, k)
    bp = pad(b, k)

    A_le_B = all(ap[i] <= bp[i] + tol for i in range(k))
    B_le_A = all(bp[i] <= ap[i] + tol for i in range(k))

    if A_le_B and B_le_A: return 0
    if A_le_B and not B_le_A: return -1
    if B_le_A and not A_le_B: return +1
    return None

def compare_zhang_yang(A: List[float], B: List[float], tol: float = 0.0) -> Optional[int]:
    a = sorted(A); b = sorted(B)
    n, m = len(a), len(b)
    k = min(n, m)

    a_sup = a[n-k:]
    b_inf = b[:k]
    A_le_B = all(a_sup[i] <= b_inf[i] + tol for i in range(k))

    b_sup = b[m-k:]
    a_inf = a[:k]
    B_le_A = all(b_sup[i] <= a_inf[i] + tol for i in range(k))

    if A_le_B and B_le_A: return 0
    if A_le_B and not B_le_A: return -1
    if B_le_A and not A_le_B: return +1
    return None

def compare_beta_gcd_scan(A: List[float], B: List[float], wfam: WeightFamily, tol: float = 0.0) -> Optional[int]:
    a = sorted(A); b = sorted(B)
    n, m = len(a), len(b)
    wa = wfam(n)
    wb = wfam(m)
    Sa, Sb = int(sum(wa)), int(sum(wb))
    g = math.gcd(Sa, Sb)
    alpha = Sb // g
    beta  = Sa // g

    ca = [int(alpha * wa[i]) for i in range(n)]
    cb = [int(beta  * wb[j]) for j in range(m)]
    L = sum(ca)

    def leq_block(xvals, xcnts, yvals, ycnts) -> bool:
        i = j = 0
        rx = xcnts[0]
        ry = ycnts[0]
        remaining = L
        while remaining > 0:
            if xvals[i] > yvals[j] + tol:
                return False
            step = rx if rx < ry else ry
            rx -= step
            ry -= step
            remaining -= step
            if rx == 0:
                i += 1
                if i < len(xvals):
                    rx = xcnts[i]
            if ry == 0:
                j += 1
                if j < len(yvals):
                    ry = ycnts[j]
        return True

    A_le_B = leq_block(a, ca, b, cb)
    B_le_A = leq_block(b, cb, a, ca)

    if A_le_B and B_le_A: return 0
    if A_le_B and not B_le_A: return -1
    if B_le_A and not A_le_B: return +1
    return None

# --------------------------------
# Strategy engine
# --------------------------------
@dataclass
class StrategyResult:
    name: str
    daily_ret: pd.Series
    equity: pd.Series
    turnover: float
    incomparables: int
    pos: pd.Series

def choose_rho_from_data(df_prices: pd.DataFrame, floor: float = 0.02, cap: float = 0.20) -> float:
    ret = df_prices.pct_change().dropna()
    neg = (-ret).clip(lower=0.0).values.ravel()
    neg = neg[np.isfinite(neg)]
    if len(neg) < 50:
        return 0.05
    q = float(np.quantile(neg, 0.95))
    return min(cap, max(floor, q))

def build_stress_from_prices(df_prices: pd.DataFrame, rho: float) -> pd.DataFrame:
    ret = df_prices.pct_change().dropna()
    stress = clip01((-ret.values) / rho)
    return pd.DataFrame(stress, index=ret.index, columns=ret.columns)

def perf_summary_full(daily_ret: pd.Series,
                      turnover: float,
                      incomparables: int,
                      ann_factor: float) -> Dict[str, float]:
    r = daily_ret.dropna()
    N = len(r)
    if N < 5:
        return {k: np.nan for k in ["Ann.Return","Ann.Vol","Sharpe","Sortino","Calmar","MaxDD","VaR5","CVaR5",
                                    "HitRate","Turnover","TurnoverAnn","IncompDays","Ndays",
                                    "NW_t(mean)","NW_se(mean)","AlphaAnn","tAlphaNW","Beta"]}

    eq = (1.0 + r).cumprod()
    cagr = float(eq.iloc[-1] ** (ann_factor / N) - 1.0)

    mu = float(r.mean())
    sig = float(r.std(ddof=1))
    ann_vol = sig * math.sqrt(ann_factor) if sig > 0 else 0.0
    sharpe = (mu / sig) * math.sqrt(ann_factor) if sig > 0 else 0.0

    dd = max_drawdown(eq)
    calmar = (cagr / abs(dd)) if (dd < 0) else np.nan

    ddv = downside_dev(r.values)
    sortino = (mu / ddv) * math.sqrt(ann_factor) if (ddv and ddv > 0) else np.nan

    var5, cvar5 = var_cvar(r.values, alpha=0.05)

    hit = float((r > 0).mean())
    t_mean, se_mean = nw_hac_tstat(r.values)

    return {
        "Ann.Return": cagr,
        "Ann.Vol": ann_vol,
        "Sharpe": sharpe,
        "Sortino": float(sortino),
        "Calmar": float(calmar),
        "MaxDD": dd,
        "VaR5": float(var5),
        "CVaR5": float(cvar5),
        "HitRate": hit,
        "Turnover": float(turnover),
        "TurnoverAnn": annualize_turnover(turnover, N, ann_factor),
        "IncompDays": int(incomparables),
        "Ndays": int(N),
        "NW_t(mean)": float(t_mean),
        "NW_se(mean)": float(se_mean),
    }

def run_switching_range(df_prices: pd.DataFrame,
                        name_a: str, name_b: str,
                        H: int, lam: float,
                        probs_a: Tuple[float, ...], probs_b: Tuple[float, ...],
                        rho: float,
                        tcost_bps: float,
                        slip_bps: float,
                        dedup: bool,
                        eval_start: Optional[pd.Timestamp],
                        eval_end: Optional[pd.Timestamp],
                        tol: float = 1e-12) -> Tuple[List[StrategyResult], pd.DataFrame, Dict[str, float]]:

    series_a, series_b = df_prices.columns[0], df_prices.columns[1]
    ret = df_prices.pct_change().dropna()
    stress = build_stress_from_prices(df_prices, rho)

    dates = ret.index

    min_tpos = H - 1
    max_tpos = len(dates) - 2
    if max_tpos <= min_tpos:
        raise ValueError("Not enough data for chosen H.")



    if eval_start is None:
        eval_start = dates[min_tpos]
    if eval_end is None:
        eval_end = dates[max_tpos]
    eval_start = pd.to_datetime(eval_start)
    eval_end = pd.to_datetime(eval_end)




    tpos_all = np.arange(min_tpos, max_tpos + 1)
    tpos_eval = [t for t in tpos_all if (dates[t] >= eval_start and dates[t] <= eval_end)]
    if len(tpos_eval) == 0:
        raise ValueError("Empty evaluation range after alignment.")

    def thfe_at(col: str, idx_pos: int, probs: Tuple[float, ...]) -> List[float]:
        window = stress[col].iloc[idx_pos - H + 1: idx_pos + 1].values[::-1]
        return make_thfe(window, lam, probs, dedup=dedup)

    methods: List[Tuple[str, Callable[[List[float], List[float]], Optional[int]]]] = [
        ("Uniform lcm ($w\\equiv 1$)", lambda A, B: compare_beta_gcd_scan(A, B, w_uniform, tol=tol)),
        ("Centered (triangular)",     lambda A, B: compare_beta_gcd_scan(A, B, w_centered_triangular, tol=tol)),
        ("Zhang--Yang order",         lambda A, B: compare_zhang_yang(A, B, tol=tol)),
        ("Optimistic padding",        lambda A, B: compare_padding(A, B, "optimistic", tol=tol)),
        ("Pessimistic padding",       lambda A, B: compare_padding(A, B, "pessimistic", tol=tol)),
        ("Tail-dominant $w_i=n^{i-1}$", lambda A, B: compare_beta_gcd_scan(A, B, w_tail_base_n, tol=tol)),
    ]

    def pos_to_weights(pos: str) -> Tuple[float, float]:
        if pos == "A": return (1.0, 0.0)
        if pos == "B": return (0.0, 1.0)
        if pos == "EW": return (0.5, 0.5)
        raise ValueError("bad pos")




    def build_bench(pos_index: pd.DatetimeIndex) -> List[StrategyResult]:
        rets_next = ret.loc[pos_index][[series_a, series_b]].copy()

        bh_a = pd.Series(rets_next[series_a].values, index=pos_index, name="BH_A")
        bh_b = pd.Series(rets_next[series_b].values, index=pos_index, name="BH_B")

        ew_rebal = 0.5 * rets_next[series_a].values + 0.5 * rets_next[series_b].values
        ew_rebal = pd.Series(ew_rebal, index=pos_index, name="EW_REBAL")

        eq_a = (1.0 + bh_a).cumprod()
        eq_b = (1.0 + bh_b).cumprod()
        eq_ew_bh = 0.5 * eq_a + 0.5 * eq_b
        ew_bh = eq_ew_bh.pct_change().fillna(0.0)
        ew_bh = pd.Series(ew_bh.values, index=pos_index, name="EW_BUYHOLD")

        return [
            StrategyResult(f"BH_{name_a}", bh_a, (1.0 + bh_a).cumprod(), 0.0, 0, pd.Series("A", index=pos_index)),
            StrategyResult(f"BH_{name_b}", bh_b, (1.0 + bh_b).cumprod(), 0.0, 0, pd.Series("B", index=pos_index)),
            StrategyResult("EW_50_50_REBAL", ew_rebal, (1.0 + ew_rebal).cumprod(), 0.0, 0, pd.Series("EW", index=pos_index)),
            StrategyResult("EW_50_50_BUYHOLD", ew_bh, (1.0 + ew_bh).cumprod(), 0.0, 0, pd.Series("EW", index=pos_index)),
        ]

    def backtest_one(method_name: str, cmpfun) -> StrategyResult:
        pos = []
        incomps = 0
        turnovers = 0.0
        prev_w = (0.5, 0.5)



        for tpos in tpos_eval:
            A = thfe_at(series_a, tpos, probs_a)
            B = thfe_at(series_b, tpos, probs_b)
            c = cmpfun(A, B)

            if c is None or c == 0:
                if c is None:
                    incomps += 1
                pos_t = "EW"
            elif c < 0:
                pos_t = "A"
            else:
                pos_t = "B"

            w = pos_to_weights(pos_t)
            tau = 0.5 * (abs(w[0] - prev_w[0]) + abs(w[1] - prev_w[1]))
            turnovers += tau
            prev_w = w
            pos.append(pos_t)

        pos_index = pd.DatetimeIndex([dates[t] for t in tpos_eval])
        pos_s = pd.Series(pos, index=pos_index, name="pos")



        next_dates = pd.DatetimeIndex([dates[t+1] for t in tpos_eval])
        rets_next = ret.loc[next_dates][[series_a, series_b]].copy()
        rets_next.index = pos_index
        wA = pos_s.map(lambda z: pos_to_weights(z)[0]).astype(float)
        wB = pos_s.map(lambda z: pos_to_weights(z)[1]).astype(float)
        gross = wA.values * rets_next[series_a].values + wB.values * rets_next[series_b].values

        kappa = (tcost_bps + slip_bps) / 10000.0



        prev_w = (0.5, 0.5)
        tau_list = []
        for z in pos:
            w = pos_to_weights(z)
            tau = 0.5 * (abs(w[0] - prev_w[0]) + abs(w[1] - prev_w[1]))
            tau_list.append(tau)
            prev_w = w
        tau_s = pd.Series(tau_list, index=pos_index)

        net = (1.0 + gross) * (1.0 - kappa * tau_s.values) - 1.0

        net_s = pd.Series(net, index=pos_index, name="ret")
        eq = (1.0 + net_s).cumprod()
        return StrategyResult(method_name, net_s, eq, turnovers, incomps, pos_s)

    results = [backtest_one(nm, fn) for nm, fn in methods]

    pos_index = results[0].daily_ret.index
    bench = build_bench(pos_index)
    results_all = results + bench

    ann_factor = estimate_ann_factor(pos_index)
    rows = []
    for r in results_all:
        s = perf_summary_full(r.daily_ret, r.turnover, r.incomparables, ann_factor=ann_factor)

        bench_ret = [b for b in results_all if b.name == "EW_50_50_BUYHOLD"][0].daily_ret.reindex(r.daily_ret.index)
        ols = nw_ols_alpha_beta(r.daily_ret.values, bench_ret.values, add_const=True)
        s["AlphaAnn"] = float(ols["alpha"]) * ann_factor
        s["tAlphaNW"] = float(ols["t_alpha"])
        s["Beta"] = float(ols["beta"])
        rows.append((r.name, s))
    summary = pd.DataFrame({k: v for k, v in rows}).T

    diag = {
        "rho": float(rho),
        "dedup": bool(dedup),
        "ann_factor": float(ann_factor),
        "eval_start": str(eval_start.date()),
        "eval_end": str(eval_end.date()),
        "tcost_bps": float(tcost_bps),
        "slip_bps": float(slip_bps),
        "H": int(H),
        "lam": float(lam),
    }
    return results_all, summary, diag

# -------------------------------------------------
# Bootstrap por bloques + Reality Check (White)
# ------------------------------------------------------
def moving_block_bootstrap_indices(n: int, block_len: int, rng: np.random.Generator) -> np.ndarray:
    if n <= 1:
        return np.arange(n)
    block_len = max(1, int(block_len))
    out = []
    while len(out) < n:
        start = rng.integers(0, n)
        block = [(start + k) % n for k in range(block_len)]
        out.extend(block)
    return np.array(out[:n], dtype=int)

def bootstrap_ci_metrics(r: np.ndarray, ann_factor: float, B: int = 1000, block_len: Optional[int] = None, seed: int = 123) -> Dict[str, Tuple[float,float,float]]:
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    n = len(r)
    if n < 50:
        return {}
    if block_len is None:
        block_len = max(5, int(round(ann_factor / 12.0)))  # ~1 mes
    rng = np.random.default_rng(seed)

    def metrics(x):
        x = np.asarray(x, float)
        eq = np.cumprod(1.0 + x)
        cagr = float(eq[-1] ** (ann_factor / len(x)) - 1.0)
        mu = float(np.mean(x))
        sig = float(np.std(x, ddof=1))
        sharpe = (mu / sig) * math.sqrt(ann_factor) if sig > 0 else np.nan
        return mu, sharpe, cagr

    mu0, sh0, cg0 = metrics(r)
    mu_s, sh_s, cg_s = [], [], []
    for b in range(B):
        idx = moving_block_bootstrap_indices(n, block_len, rng)
        xb = r[idx]
        mu1, sh1, cg1 = metrics(xb)
        mu_s.append(mu1); sh_s.append(sh1); cg_s.append(cg1)

    def ci(arr, point):
        a = np.asarray(arr, float)
        lo = float(np.quantile(a, 0.025))
        hi = float(np.quantile(a, 0.975))
        return (float(point), lo, hi)

    return {
        "mean": ci(mu_s, mu0),
        "Sharpe": ci(sh_s, sh0),
        "CAGR": ci(cg_s, cg0),
        "block_len": (float(block_len), float(block_len), float(block_len)),
        "B": (float(B), float(B), float(B)),
    }

def white_reality_check(diffs: np.ndarray, ann_factor: float, B: int = 1000, block_len: Optional[int] = None, seed: int = 123) -> Dict[str, float]:

    diffs = np.asarray(diffs, float)
    diffs = diffs[np.all(np.isfinite(diffs), axis=1)]
    T, M = diffs.shape
    if T < 80 or M < 1:
        return dict(T_obs=np.nan, pval=np.nan, block_len=np.nan, B=np.nan)

    if block_len is None:
        block_len = max(5, int(round(ann_factor / 12.0)))

    dbar = diffs.mean(axis=0, keepdims=True)
    dc = diffs - dbar  # centrar
    T_obs = float(np.max(dbar))

    rng = np.random.default_rng(seed)
    Tb = []
    for b in range(B):
        idx = moving_block_bootstrap_indices(T, block_len, rng)
        sample = dc[idx, :]
        means = sample.mean(axis=0)
        Tb.append(float(np.max(means)))
    Tb = np.asarray(Tb, float)
    pval = float((1.0 + np.sum(Tb >= T_obs)) / (B + 1.0))

    return dict(T_obs=T_obs, pval=pval, block_len=float(block_len), B=float(B))

# ----------------------------------
# Walk-forward (rolling) OOS
# ----------------------------------
def walk_forward_oos(df_prices: pd.DataFrame,
                     name_a: str, name_b: str,
                     H: int, lam: float,
                     probs_a: Tuple[float, ...], probs_b: Tuple[float, ...],
                     tcost_bps: float, slip_bps: float,
                     dedup: bool,
                     train_years: int = 5,
                     test_years: int = 1,
                     floor_rho: float = 0.02, cap_rho: float = 0.20,
                     tol: float = 1e-12) -> Tuple[List[StrategyResult], pd.DataFrame, Dict[str, float]]:


    prices = df_prices.dropna().copy()
    ret = prices.pct_change().dropna()
    dates = ret.index
    years = sorted(pd.unique(dates.year))
    if len(years) < (train_years + test_years + 1):

        rho = choose_rho_from_data(prices, floor=floor_rho, cap=cap_rho)
        res_all, summ, diag = run_switching_range(prices, name_a, name_b, H, lam, probs_a, probs_b,
                                                  rho=rho, tcost_bps=tcost_bps, slip_bps=slip_bps,
                                                  dedup=dedup, eval_start=None, eval_end=None, tol=tol)
        diag["wf_folds"] = 0
        diag["wf_note"] = "fallback_full_sample"
        return res_all, summ, diag

    fold_results: Dict[str, List[pd.Series]] = {}
    fold_pos: Dict[str, List[pd.Series]] = {}
    fold_turn: Dict[str, float] = {}
    fold_incomp: Dict[str, int] = {}

    wf_count = 0
    for i in range(0, len(years) - (train_years + test_years) + 1, test_years):
        y_train_start = years[i]
        y_train_end = years[i + train_years - 1]
        y_test_start = years[i + train_years]
        y_test_end = years[i + train_years + test_years - 1]

        train_start = pd.Timestamp(f"{y_train_start}-01-01")
        train_end   = pd.Timestamp(f"{y_train_end}-12-31")
        test_start  = pd.Timestamp(f"{y_test_start}-01-01")
        test_end    = pd.Timestamp(f"{y_test_end}-12-31")


        if test_start < dates.min() or test_start > dates.max():
            continue

        train_prices = prices.loc[(prices.index >= train_start) & (prices.index <= train_end)]
        if len(train_prices) < 80:
            continue

        rho = choose_rho_from_data(train_prices, floor=floor_rho, cap=cap_rho)



        try:
            res_all, summ, diag = run_switching_range(
                prices, name_a, name_b,
                H=H, lam=lam,
                probs_a=probs_a, probs_b=probs_b,
                rho=rho,
                tcost_bps=tcost_bps,
                slip_bps=slip_bps,
                dedup=dedup,
                eval_start=test_start,
                eval_end=test_end,
                tol=tol
            )
        except Exception:
            continue

        wf_count += 1
        for r in res_all:
            fold_results.setdefault(r.name, []).append(r.daily_ret)
            fold_pos.setdefault(r.name, []).append(r.pos)
            fold_turn[r.name] = fold_turn.get(r.name, 0.0) + float(r.turnover)
            fold_incomp[r.name] = fold_incomp.get(r.name, 0) + int(r.incomparables)

    if wf_count == 0:
        rho = choose_rho_from_data(prices, floor=floor_rho, cap=cap_rho)
        res_all, summ, diag = run_switching_range(prices, name_a, name_b, H, lam, probs_a, probs_b,
                                                  rho=rho, tcost_bps=tcost_bps, slip_bps=slip_bps,
                                                  dedup=dedup, eval_start=None, eval_end=None, tol=tol)
        diag["wf_folds"] = 0
        diag["wf_note"] = "fallback_full_sample_after_empty_wf"
        return res_all, summ, diag

    results_all = []
    all_names = sorted(fold_results.keys())
    for nm in all_names:
        rcat = pd.concat(fold_results[nm], axis=0).sort_index()

        rcat = rcat[~rcat.index.duplicated(keep="first")]
        eq = (1.0 + rcat).cumprod()

        pcat = pd.concat(fold_pos[nm], axis=0).sort_index()
        pcat = pcat[~pcat.index.duplicated(keep="first")]
        results_all.append(StrategyResult(nm, rcat, eq, fold_turn.get(nm, 0.0), fold_incomp.get(nm, 0), pcat))

    ann_factor = estimate_ann_factor(results_all[0].daily_ret.index)
    rows = []
    for r in results_all:
        s = perf_summary_full(r.daily_ret, r.turnover, r.incomparables, ann_factor=ann_factor)

        bench = [x for x in results_all if x.name == "EW_50_50_BUYHOLD"]
        if len(bench) == 1:
            bench_ret = bench[0].daily_ret.reindex(r.daily_ret.index)
            ols = nw_ols_alpha_beta(r.daily_ret.values, bench_ret.values, add_const=True)
            s["AlphaAnn"] = float(ols["alpha"]) * ann_factor
            s["tAlphaNW"] = float(ols["t_alpha"])
            s["Beta"] = float(ols["beta"])
        else:
            s["AlphaAnn"] = np.nan
            s["tAlphaNW"] = np.nan
            s["Beta"] = np.nan
        rows.append((r.name, s))
    summary = pd.DataFrame({k: v for k, v in rows}).T

    diag = {
        "wf_folds": wf_count,
        "ann_factor": float(ann_factor),
        "H": int(H),
        "lam": float(lam),
        "dedup": bool(dedup),
        "tcost_bps": float(tcost_bps),
        "slip_bps": float(slip_bps),
    }
    return results_all, summary, diag

# ----------------------------------
# Plot utils
# ----------------------------------
def plot_equities(results: List[StrategyResult], title: str, outpath: str) -> None:
    plt.figure(figsize=(10, 6))
    for r in results:
        plt.plot(r.equity.index, r.equity.values, label=r.name, linewidth=1.1)
    plt.title(title)
    plt.yscale("log")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(outpath, dpi=160)
    plt.close()

def plot_heatmap(mat: np.ndarray, xlabels: List[str], ylabels: List[str], title: str, outpath: str) -> None:
    plt.figure(figsize=(10, 6))
    plt.imshow(mat, aspect="auto")
    plt.xticks(range(len(xlabels)), xlabels, rotation=0)
    plt.yticks(range(len(ylabels)), ylabels)
    plt.colorbar()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=160)
    plt.close()

# -----------------------------
# Scenarios
# -----------------------------
GRID_SCENARIOS: Dict[str, Tuple[Tuple[float, ...], Tuple[float, ...]]] = {
    "hetero_baseline": ((0.25, 0.50, 0.75), (0.20, 0.35, 0.45, 0.60, 0.85)),
    "hetero_tighter":  ((0.30, 0.50, 0.70), (0.25, 0.40, 0.50, 0.60, 0.80)),
    "homo_3q":         ((0.25, 0.50, 0.75), (0.25, 0.50, 0.75)),
}

SUBPERIODS = [
    ("2010-2014", "2010-01-01", "2014-12-31"),
    ("2015-2019", "2015-01-01", "2019-12-31"),
    ("2020-2021", "2020-01-01", "2021-12-31"),
    ("2022-2025", "2022-01-01", "2025-12-31"),
]

# -------------
# MAIN
# --------------



def main(argv: Optional[List[str]] = None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--start", type=str, default="2010-01-01")
    ap.add_argument("--end", type=str, default="2025-12-31")

    # baseline params
    ap.add_argument("--H", type=int, default=10)
    ap.add_argument("--lam", type=float, default=0.94)
    ap.add_argument("--dedup", action="store_true", default=True)
    ap.add_argument("--no-dedup", dest="dedup", action="store_false")
    ap.add_argument("--scenario", type=str, default="hetero_baseline")

    # costs
    ap.add_argument("--tcosts", type=str, default="0,5,10,25")
    ap.add_argument("--slippage", type=str, default="0,5")  # bps adicionales proporcionales a turnover

    # walk-forward
    ap.add_argument("--wf_train_years", type=int, default=5)
    ap.add_argument("--wf_test_years", type=int, default=1)

    # stats
    ap.add_argument("--boot_B", type=int, default=800)
    ap.add_argument("--boot_seed", type=int, default=123)

    # robustness grid sizes (moderado)
    ap.add_argument("--grid_H", type=str, default="5,10,20")
    ap.add_argument("--grid_lam", type=str, default="0.90,0.94,0.97")
    ap.add_argument("--grid_rhomult", type=str, default="0.8,1.0,1.2")
    ap.add_argument("--grid_dedup", type=str, default="1,0")  # 1=True,0=False
    ap.add_argument("--grid_scen", type=str, default="hetero_baseline,hetero_tighter")

    ap.add_argument("--outdir", type=str, default="out_q1_plus")
    ap.add_argument("--seed", type=int, default=123)
    args = _parse_known_notebook_safe(ap, argv)

    os.makedirs(args.outdir, exist_ok=True)
    np.random.seed(args.seed)


    tcost_list = [float(x.strip()) for x in args.tcosts.split(",") if x.strip() != ""]
    slip_list = [float(x.strip()) for x in args.slippage.split(",") if x.strip() != ""]
    if args.scenario not in GRID_SCENARIOS:
        raise ValueError(f"scenario desconocido: {args.scenario}. Opciones: {list(GRID_SCENARIOS.keys())}")
    probs_a_base, probs_b_base = GRID_SCENARIOS[args.scenario]

    gold_series = pick_gold_series(args.start, args.end)
    if gold_series is None:
        print("[warn] No se pudo descargar ORO desde FRED. Se saltan pares con ORO.")
    else:
        print(f"[data] GOLD series selected: {gold_series}")

    # === PAIRS ===
    pairs = [
        ("CBBTCUSD", "CBETHUSD", "BTC", "ETH"),
        ("SP500", "NASDAQCOM", "SP500", "NASDAQ"),
        ("CBBTCUSD", "SP500", "BTC", "SP500"),
        ("CBBTCUSD", "NASDAQCOM", "BTC", "NASDAQ"),
        ("CBETHUSD", "SP500", "ETH", "SP500"),
        ("CBETHUSD", "NASDAQCOM", "ETH", "NASDAQ"),
    ]
    if gold_series is not None:
        pairs += [
            (gold_series, "SP500", "GOLD", "SP500"),
            (gold_series, "CBBTCUSD", "GOLD", "BTC"),
        ]

    # robust grid
    grid_H = [int(x) for x in args.grid_H.split(",") if x.strip() != ""]
    grid_lam = [float(x) for x in args.grid_lam.split(",") if x.strip() != ""]
    grid_rhomult = [float(x) for x in args.grid_rhomult.split(",") if x.strip() != ""]
    grid_dedup = [bool(int(x)) for x in args.grid_dedup.split(",") if x.strip() != ""]
    grid_scen = [x.strip() for x in args.grid_scen.split(",") if x.strip() != ""]
    for sc in grid_scen:
        if sc not in GRID_SCENARIOS:
            raise ValueError(f"grid_scen contiene escenario desconocido: {sc}")

    # Parámetros de bootstrap
    B = int(args.boot_B)
    boot_seed = int(args.boot_seed)

    print("\n=========================")
    print("[info] Configuración")
    print("=========================")
    print(f"start={args.start} end={args.end} outdir={args.outdir}")
    print(f"baseline: H={args.H} lam={args.lam} dedup={args.dedup} scenario={args.scenario}")
    print(f"tcosts(bps)={tcost_list} slippage(bps)={slip_list}")
    print(f"WF: train_years={args.wf_train_years} test_years={args.wf_test_years}")
    print(f"Bootstrap: B={B} seed={boot_seed}")
    print(f"Grid: H={grid_H} lam={grid_lam} rhomult={grid_rhomult} dedup={grid_dedup} scen={grid_scen}")
    print("Timing: decisión al cierre de t, ejecución/retorno aplicado en t+1.\n")


    # Directorios
    out_pairs_dir = os.path.join(args.outdir, "pairs")
    out_grid_dir = os.path.join(args.outdir, "robustness")
    os.makedirs(out_pairs_dir, exist_ok=True)
    os.makedirs(out_grid_dir, exist_ok=True)


    # -----------------------------
    # Loop pairs
    # -----------------------------
    for (sa, sb, na, nb) in pairs:
        print("\n" + "="*90)
        print(f"[pair] {na} vs {nb}  ({sa} vs {sb})")
        print("="*90)
        try:
            dfp = load_pair_prices(sa, sb, args.start, args.end)
        except Exception as e:
            print(f"[skip] descarga fallida para {sa}/{sb}: {e}")
            continue

        dfp = dfp[[sa, sb]].dropna()
        if len(dfp) < 250:
            print("[skip] not enough data after alignment.")
            continue

        # ann_factor
        ret = dfp.pct_change().dropna()
        ann_guess = estimate_ann_factor(ret.index)
        print(f"[data] aligned obs={len(dfp)} | returns obs={len(ret)} | ann_factor_est≈{ann_guess:.1f}")

        # rho baseline (full sample)
        rho_full = choose_rho_from_data(dfp)
        print(f"[param] rho_full={rho_full:.5f}")


        # =========================---------------
        # (A) BASELINE IN-SAMPLE FULL PERIOD
        # =========================---------------
        print("\n[A] Baseline IN-SAMPLE (full period) — tablas + curvas equity")
        for tcost in tcost_list:
            for slip in slip_list:
                tag = f"{safe_name(na)}_{safe_name(nb)}__IS__{args.scenario}__H{args.H}_lam{args.lam}_dedup{int(args.dedup)}__tc{int(tcost)}_sl{int(slip)}"
                try:
                    results_all, summary, diag = run_switching_range(
                        dfp, na, nb,
                        H=args.H, lam=args.lam,
                        probs_a=probs_a_base, probs_b=probs_b_base,
                        rho=rho_full,
                        tcost_bps=tcost, slip_bps=slip,
                        dedup=args.dedup,
                        eval_start=None, eval_end=None
                    )
                except Exception as e:
                    print(f"  [skip IS] {tag}: {e}")
                    continue

                # guardar
                out_csv = os.path.join(out_pairs_dir, f"summary_{tag}.csv")
                summary.to_csv(out_csv)

                out_png = os.path.join(out_pairs_dir, f"equity_{tag}.png")
                plot_equities(results_all, title=f"{na} vs {nb} | IS full | tc={tcost}bps slip={slip}bps | ann≈{diag['ann_factor']:.0f}",
                              outpath=out_png)

                # imprimir top
                tmp = summary[["Sharpe","Sortino","Ann.Return","MaxDD","Calmar","TurnoverAnn","IncompDays","AlphaAnn","tAlphaNW"]].copy()
                tmp["final_equity"] = [float(x.equity.iloc[-1]) for x in results_all]
                tmp = tmp.sort_values("final_equity", ascending=False)
                print(f"\n  [IS] tc={tcost}bps slip={slip}bps | ann≈{diag['ann_factor']:.0f} | eval={diag['eval_start']}..{diag['eval_end']}")
                print(tmp.head(8).to_string())

        # =========================---------------
        # (B) SUBPERIODS (IN-SAMPLE)
        # =========================--------------------
        print("\n[B] Subperiodos (IN-SAMPLE por tramos) — baseline params (tc=10, slip=0 por defecto)")
        tcost_sub = 10.0 if 10.0 in tcost_list else (tcost_list[0] if len(tcost_list) else 0.0)
        slip_sub = 0.0 if 0.0 in slip_list else (slip_list[0] if len(slip_list) else 0.0)


        sub_rows = []
        for (lbl, s0, s1) in SUBPERIODS:
            s0t, s1t = pd.Timestamp(s0), pd.Timestamp(s1)
            if s1t < ret.index.min() or s0t > ret.index.max():
                continue
            # rho por subperiodo
            df_sub = dfp.loc[(dfp.index >= s0t) & (dfp.index <= s1t)].dropna()
            if len(df_sub) < 150:
                continue
            rho_sub = choose_rho_from_data(df_sub)

            tag = f"{safe_name(na)}_{safe_name(nb)}__SUB_{lbl}__{args.scenario}__H{args.H}_lam{args.lam}_dedup{int(args.dedup)}__tc{int(tcost_sub)}_sl{int(slip_sub)}"
            try:
                results_all, summary, diag = run_switching_range(
                    dfp, na, nb,
                    H=args.H, lam=args.lam,
                    probs_a=probs_a_base, probs_b=probs_b_base,
                    rho=rho_sub,
                    tcost_bps=tcost_sub, slip_bps=slip_sub,
                    dedup=args.dedup,
                    eval_start=s0t, eval_end=s1t
                )
            except Exception as e:
                print(f"  [skip SUB] {lbl}: {e}")
                continue

            out_csv = os.path.join(out_pairs_dir, f"summary_{tag}.csv")
            summary.to_csv(out_csv)


            # guardar ranking compacto (método ganador)
            tmp = summary[["Sharpe","Ann.Return","MaxDD","TurnoverAnn","IncompDays"]].copy()
            tmp["final_equity"] = [float(x.equity.iloc[-1]) for x in results_all]
            tmp = tmp.sort_values("final_equity", ascending=False)
            winner = tmp.index[0]
            sub_rows.append({
                "pair": f"{na}-{nb}",
                "period": lbl,
                "ann": diag["ann_factor"],
                "rho": rho_sub,
                "winner": winner,
                "winner_final_equity": float(tmp.iloc[0]["final_equity"]),
                "winner_Sharpe": float(tmp.iloc[0]["Sharpe"]),
                "winner_TurnoverAnn": float(tmp.iloc[0]["TurnoverAnn"]),
                "winner_IncompDays": float(tmp.iloc[0]["IncompDays"]),
            })

            print(f"\n  [SUB {lbl}] ann≈{diag['ann_factor']:.0f} rho={rho_sub:.5f} tc={tcost_sub} slip={slip_sub} | winner={winner}")
            print(tmp.head(6).to_string())

        if len(sub_rows) > 0:
            df_subsum = pd.DataFrame(sub_rows)
            out_csv = os.path.join(out_pairs_dir, f"subperiod_summary_{safe_name(na)}_{safe_name(nb)}.csv")
            df_subsum.to_csv(out_csv, index=False)

        # ===========================================================================
        # (C) WALK-FORWARD OUT-OF-SAMPLE + Bootstrap + White Reality Check
        # ===========================================================================
        print("\n[C] Walk-forward OUT-OF-SAMPLE (rolling) + bootstrap CI + White Reality Check")

        slip_oos = 0.0 if 0.0 in slip_list else (slip_list[0] if len(slip_list) else 0.0)

        for tcost in tcost_list:
            tag = f"{safe_name(na)}_{safe_name(nb)}__OOS_WF__{args.scenario}__H{args.H}_lam{args.lam}_dedup{int(args.dedup)}__tc{int(tcost)}_sl{int(slip_oos)}"
            try:
                results_oos, summary_oos, diag_oos = walk_forward_oos(
                    dfp, na, nb,
                    H=args.H, lam=args.lam,
                    probs_a=probs_a_base, probs_b=probs_b_base,
                    tcost_bps=tcost, slip_bps=slip_oos,
                    dedup=args.dedup,
                    train_years=args.wf_train_years,
                    test_years=args.wf_test_years
                )
            except Exception as e:
                print(f"  [skip OOS] tc={tcost}: {e}")
                continue

            out_csv = os.path.join(out_pairs_dir, f"summary_{tag}.csv")
            summary_oos.to_csv(out_csv)

            out_png = os.path.join(out_pairs_dir, f"equity_{tag}.png")
            plot_equities(results_oos, title=f"{na} vs {nb} | OOS WF | tc={tcost} slip={slip_oos} | folds={diag_oos['wf_folds']} | ann≈{diag_oos['ann_factor']:.0f}",
                          outpath=out_png)

            # ranking
            tmp = summary_oos[["Sharpe","Sortino","Ann.Return","MaxDD","Calmar","TurnoverAnn","IncompDays","AlphaAnn","tAlphaNW"]].copy()
            tmp["final_equity"] = [float(x.equity.iloc[-1]) for x in results_oos]
            tmp = tmp.sort_values("final_equity", ascending=False)

            print(f"\n  [OOS WF] tc={tcost} slip={slip_oos} | folds={diag_oos['wf_folds']} | ann≈{diag_oos['ann_factor']:.0f}")
            print(tmp.head(8).to_string())

            # Bootstrap CI del método ganador
            winner_name = tmp.index[0]
            winner = [r for r in results_oos if r.name == winner_name][0]
            ann_factor = float(diag_oos["ann_factor"])
            ci = bootstrap_ci_metrics(winner.daily_ret.values, ann_factor=ann_factor, B=B, seed=boot_seed)

            if ci:
                print("\n    [bootstrap CI 95%] (point, lo, hi)")
                for k in ["mean","Sharpe","CAGR"]:
                    if k in ci:
                        print(f"    {k:>6s}: {ci[k]}  (block_len≈{ci['block_len'][0]:.0f}, B={int(ci['B'][0])})")


            # White Reality Check vs benchmark EW_BUYHOLD
            bench = [r for r in results_oos if r.name == "EW_50_50_BUYHOLD"]
            if len(bench) == 1:
                bench_r = bench[0].daily_ret.reindex(winner.daily_ret.index).values
                # diffs para TODOS los métodos "propuestos" (no benchmarks)
                proposed = [r for r in results_oos if r.name not in ["BH_"+na, "BH_"+nb, "EW_50_50_REBAL", "EW_50_50_BUYHOLD"]]
                if len(proposed) >= 2:
                    D = []
                    idx = winner.daily_ret.index
                    for r in proposed:
                        rr = r.daily_ret.reindex(idx).values
                        D.append(rr - bench_r)
                    D = np.column_stack(D)
                    rc = white_reality_check(D, ann_factor=ann_factor, B=B, seed=boot_seed)
                    print(f"\n    [White Reality Check vs EW_BUYHOLD] T_obs(max mean diff)={rc['T_obs']:.6f} | p-value={rc['pval']:.4f} | block_len≈{rc['block_len']:.0f} | B={int(rc['B'])}")
                else:
                    print("\n    [White Reality Check] (skip) no hay suficientes métodos propuestos.")
            else:
                print("\n    [White Reality Check] (skip) no se encontró EW_50_50_BUYHOLD en resultados OOS.")

        # =================================================================
        # (D) ROBUSTEZ (GRID) sobre OOS, tc=10 , slip=0
        # =================================================================
        print("\n[D] Robustez (grid) sobre OOS — win-frequency + heatmaps (tc=10, slip=0 por defecto)")
        tcost_grid = 10.0 if 10.0 in tcost_list else (tcost_list[0] if len(tcost_list) else 0.0)
        slip_grid = 0.0 if 0.0 in slip_list else (slip_list[0] if len(slip_list) else 0.0)

        # baseline rho para multiplicadores (en train de cada fold ya se recalcula; aquí rhomult actúa sobre rho_train por fold)

        def walk_forward_oos_with_rhomult(rhomult: float, scen: str, H0: int, lam0: float, ded: bool):
            probs_a, probs_b = GRID_SCENARIOS[scen]
            prices = dfp.dropna().copy()
            retloc = prices.pct_change().dropna()
            datesloc = retloc.index
            yearsloc = sorted(pd.unique(datesloc.year))
            if len(yearsloc) < (args.wf_train_years + args.wf_test_years + 1):
                rho = choose_rho_from_data(prices)
                rho = rhomult * rho
                return run_switching_range(
                    prices, na, nb,
                    H=H0, lam=lam0,
                    probs_a=probs_a, probs_b=probs_b,
                    rho=rho,
                    tcost_bps=tcost_grid, slip_bps=slip_grid,
                    dedup=ded,
                    eval_start=None, eval_end=None
                )

            fold_results = {}
            fold_turn = {}
            fold_incomp = {}
            wf_count = 0
            for i in range(0, len(yearsloc) - (args.wf_train_years + args.wf_test_years) + 1, args.wf_test_years):
                y_train_start = yearsloc[i]
                y_train_end = yearsloc[i + args.wf_train_years - 1]
                y_test_start = yearsloc[i + args.wf_train_years]
                y_test_end = yearsloc[i + args.wf_train_years + args.wf_test_years - 1]

                train_start = pd.Timestamp(f"{y_train_start}-01-01")
                train_end   = pd.Timestamp(f"{y_train_end}-12-31")
                test_start  = pd.Timestamp(f"{y_test_start}-01-01")
                test_end    = pd.Timestamp(f"{y_test_end}-12-31")

                train_prices = prices.loc[(prices.index >= train_start) & (prices.index <= train_end)]
                if len(train_prices) < 80:
                    continue
                rho = choose_rho_from_data(train_prices)
                rho = rhomult * rho

                try:
                    res_all, summ, diag = run_switching_range(
                        prices, na, nb,
                        H=H0, lam=lam0,
                        probs_a=probs_a, probs_b=probs_b,
                        rho=rho,
                        tcost_bps=tcost_grid, slip_bps=slip_grid,
                        dedup=ded,
                        eval_start=test_start, eval_end=test_end
                    )
                except Exception:
                    continue

                wf_count += 1
                for r in res_all:
                    fold_results.setdefault(r.name, []).append(r.daily_ret)
                    fold_turn[r.name] = fold_turn.get(r.name, 0.0) + float(r.turnover)
                    fold_incomp[r.name] = fold_incomp.get(r.name, 0) + int(r.incomparables)

            if wf_count == 0:
                rho = choose_rho_from_data(prices)
                rho = rhomult * rho
                return run_switching_range(
                    prices, na, nb,
                    H=H0, lam=lam0,
                    probs_a=probs_a, probs_b=probs_b,
                    rho=rho,
                    tcost_bps=tcost_grid, slip_bps=slip_grid,
                    dedup=ded,
                    eval_start=None, eval_end=None
                )

            results_all = []
            for nm, lst in fold_results.items():
                rcat = pd.concat(lst, axis=0).sort_index()
                rcat = rcat[~rcat.index.duplicated(keep="first")]
                eq = (1.0 + rcat).cumprod()
                results_all.append(StrategyResult(nm, rcat, eq, fold_turn.get(nm, 0.0), fold_incomp.get(nm, 0), pd.Series(index=rcat.index, data="")))

            ann_factor = estimate_ann_factor(results_all[0].daily_ret.index)
            rows = []
            for r in results_all:
                s = perf_summary_full(r.daily_ret, r.turnover, r.incomparables, ann_factor=ann_factor)
                rows.append((r.name, s))
            summ = pd.DataFrame({k: v for k, v in rows}).T
            diag = {"wf_folds": wf_count, "ann_factor": ann_factor, "rhomult": rhomult, "scenario": scen, "H": H0, "lam": lam0, "dedup": ded}
            return results_all, summ, diag

        # Ejecutar grid
        grid_records = []
        win_count = {}


        # Para heatmap: guardamos Sharpe de cada método propuesto (no benchmarks) para cada (H,lam) agregando resto
        # agregación: media de Sharpe a través de rhomult, dedup, escenario

        method_names_proposed = ["Uniform lcm ($w\\equiv 1$)", "Centered (triangular)", "Zhang--Yang order",
                                 "Optimistic padding", "Pessimistic padding", "Tail-dominant $w_i=n^{i-1}$"]

        sharpe_acc = {m: {(H0, lam0): [] for H0 in grid_H for lam0 in grid_lam} for m in method_names_proposed}

        total_runs = 0
        for scen in grid_scen:
            probsA, probsB = GRID_SCENARIOS[scen]
            for H0 in grid_H:
                for lam0 in grid_lam:
                    for rm in grid_rhomult:
                        for ded in grid_dedup:
                            total_runs += 1
                            try:
                                res_g, summ_g, diag_g = walk_forward_oos_with_rhomult(rm, scen, H0, lam0, ded)
                            except Exception:
                                continue

                            tmpg = summ_g[["Sharpe","Ann.Return","MaxDD","TurnoverAnn","IncompDays"]].copy()
                            tmpg["final_equity"] = [float(x.equity.iloc[-1]) for x in res_g]
                            tmpg = tmpg.sort_values("final_equity", ascending=False)
                            winner = tmpg.index[0]
                            win_count[winner] = win_count.get(winner, 0) + 1

                            # acumular Sharpe por método propuesto
                            for m in method_names_proposed:
                                if m in summ_g.index:
                                    sharpe_acc[m][(H0, lam0)].append(float(summ_g.loc[m, "Sharpe"]))

                            grid_records.append({
                                "pair": f"{na}-{nb}",
                                "scenario": scen,
                                "H": H0,
                                "lam": lam0,
                                "rhomult": rm,
                                "dedup": int(ded),
                                "tcost_bps": tcost_grid,
                                "slip_bps": slip_grid,
                                "wf_folds": diag_g.get("wf_folds", np.nan),
                                "ann_factor": diag_g.get("ann_factor", np.nan),
                                "winner": winner,
                                "winner_final_equity": float(tmpg.iloc[0]["final_equity"]),
                                "winner_Sharpe": float(tmpg.iloc[0]["Sharpe"]),
                                "winner_TurnoverAnn": float(tmpg.iloc[0]["TurnoverAnn"]),
                                "winner_IncompDays": float(tmpg.iloc[0]["IncompDays"]),
                            })

        df_grid = pd.DataFrame(grid_records)
        out_csv = os.path.join(out_grid_dir, f"grid_results_{safe_name(na)}_{safe_name(nb)}.csv")
        df_grid.to_csv(out_csv, index=False)

        # imprimir win frequency (top)
        if len(win_count) > 0:
            wf = pd.Series(win_count).sort_values(ascending=False)
            print("\n  [robustness] Win-frequency (winner por configuración OOS del grid):")
            print(wf.head(12).to_string())
        else:
            print("\n  [robustness] (sin resultados)")

        # heatmaps Sharpe medio por (H,lam)
        for m in method_names_proposed:
            mat = np.full((len(grid_H), len(grid_lam)), np.nan, float)
            for iH, H0 in enumerate(grid_H):
                for jL, lam0 in enumerate(grid_lam):
                    vals = sharpe_acc[m][(H0, lam0)]
                    if len(vals) > 0:
                        mat[iH, jL] = float(np.mean(vals))
            out_png = os.path.join(out_grid_dir, f"heatmap_Sharpe_{safe_name(na)}_{safe_name(nb)}__{safe_name(m)}.png")
            plot_heatmap(mat,
                         xlabels=[str(x) for x in grid_lam],
                         ylabels=[str(x) for x in grid_H],
                         title=f"Mean Sharpe (OOS grid) | {na} vs {nb} | method={m} | tc={tcost_grid} slip={slip_grid}",
                         outpath=out_png)

        print(f"\n  [saved] grid CSV: {out_csv}")
        print(f"  [saved] heatmaps in: {out_grid_dir}")

    print("\n" + "="*90)
    print(f"[done] outputs written to: {args.outdir}")
    print(f"- Pair CSV/PNG: {out_pairs_dir}")
    print(f"- Robustness CSV/heatmaps: {out_grid_dir}")
    print("="*90)

# ===========================================================================
# EJECUCIÓN DIRECTA (se pueden cambiar parámetros aquí si se desea)
# ===============================================================================
main([
    "--start","2010-01-01",
    "--end","2025-12-31",
    "--H","10",
    "--lam","0.94",
    "--scenario","hetero_baseline",
    "--tcosts","0,5,10,25",
    "--slippage","0,5",
    "--wf_train_years","5",
    "--wf_test_years","1",
    "--boot_B","800",
    "--boot_seed","123",
    "--grid_H","5,10,20",
    "--grid_lam","0.90,0.94,0.97",
    "--grid_rhomult","0.8,1.0,1.2",
    "--grid_dedup","1,0",
    "--grid_scen","hetero_baseline,hetero_tighter",
    "--outdir","out_q1_plus",
])


[warn] No se pudo descargar ORO desde FRED. Se saltan pares con ORO.

[info] Configuración
start=2010-01-01 end=2025-12-31 outdir=out_q1_plus
baseline: H=10 lam=0.94 dedup=True scenario=hetero_baseline
tcosts(bps)=[0.0, 5.0, 10.0, 25.0] slippage(bps)=[0.0, 5.0]
WF: train_years=5 test_years=1
Bootstrap: B=800 seed=123
Grid: H=[5, 10, 20] lam=[0.9, 0.94, 0.97] rhomult=[0.8, 1.0, 1.2] dedup=[True, False] scen=['hetero_baseline', 'hetero_tighter']
Timing: decisión al cierre de t, ejecución/retorno aplicado en t+1.


[pair] BTC vs ETH  (CBBTCUSD vs CBETHUSD)
[data] aligned obs=3512 | returns obs=3511 | ann_factor_est≈365.0
[param] rho_full=0.06283

[A] Baseline IN-SAMPLE (full period) — tablas + curvas equity

  [IS] tc=0.0bps slip=0.0bps | ann≈365 | eval=2016-05-30..2025-12-30
                               Sharpe   Sortino  Ann.Return     MaxDD    Calmar  TurnoverAnn  IncompDays  AlphaAnn  tAlphaNW  final_equity
Pessimistic padding          1.379346  2.139028    1.091473 -0.798108  1.3675